# Create Random Data

## Patients

### Generate

In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta


def generate_patients(n=100):
    patients = []
    
    # Define date range
    start_date = datetime(1938, 1, 1)
    end_date = datetime(2008, 12, 31)
    days_between = (end_date - start_date).days

    for i in range(n):
        # 1. Sequential 4-digit ID (starting from 0000)
        patient_id = f"{i:04d}"
        
        # 2. Random gender with 55/45 split in favor of Males
        gender = random.choices(['Male', 'Female'], weights=[55, 45])[0]
        
        # 3. Random DOB within range
        random_days = random.randint(0, days_between)
        dob = (start_date + timedelta(days=random_days)).date()
        
        patients.append({
            "patient_id": patient_id,
            "gender": gender,
            "dob": dob
        })

    df = pd.DataFrame(patients)
    return df

patients_df = generate_patients(1000)
patients_df

,patient_id,gender,dob
0,0000,Male,1988-02-16
1,0001,Male,1984-04-17
2,0002,Female,1944-09-20
3,0003,Male,1974-07-30
4,0004,Male,1947-04-25
...,...,...,...
995,0995,Male,1981-12-30
996,0996,Male,1942-02-10
997,0997,Female,1990-03-19
998,0998,Female,1963-11-16


### Save

In [2]:
import os
from pathlib import Path


save_dir = Path(os.environ['HIE_DASHBOARD_BASE'], 'data', 'v_1.0')
save_dir.mkdir(parents=True, exist_ok=True)

patients_df.to_csv(Path(save_dir, 'patients.csv'), index=False)

## Diagnostic Report

In [3]:
import os
from pathlib import Path

import pandas as pd


save_dir = Path(os.environ['HIE_DASHBOARD_BASE'], 'data', 'v_1.0')
patients_file = Path(save_dir, 'patients.csv')

patients_df = pd.read_csv(patients_file)
patients_df

,patient_id,gender,dob
0,0,Male,1988-02-16
1,1,Male,1984-04-17
2,2,Female,1944-09-20
3,3,Male,1974-07-30
4,4,Male,1947-04-25
...,...,...,...
995,995,Male,1981-12-30
996,996,Male,1942-02-10
997,997,Female,1990-03-19
998,998,Female,1963-11-16


In [4]:
import pandas as pd
import numpy as np


# 1. Prepare the probabilities (must sum to 1.0)
raw_probs = [0.005, 0.95, 0.03, 0.005]
# Normalizing to ensure the sum is exactly 1.0
probs = np.array(raw_probs) / np.sum(raw_probs)

# 2. Define the corresponding values (0 to 3, matching your 4 probabilities)
values = [0, 1, 2, 3]

# 3. Generate the column for your dataframe 'df'
# (Assuming 'df' already exists)
patients_df['no_of_drs'] = np.random.choice(values, size=len(patients_df), p=probs)
patients_df

,patient_id,gender,dob,no_of_drs
0,0,Male,1988-02-16,1
1,1,Male,1984-04-17,2
2,2,Female,1944-09-20,1
3,3,Male,1974-07-30,1
4,4,Male,1947-04-25,1
...,...,...,...,...
995,995,Male,1981-12-30,1
996,996,Male,1942-02-10,1
997,997,Female,1990-03-19,2
998,998,Female,1963-11-16,1


In [5]:
patients_df['no_of_drs'].value_counts(normalize=True)

no_of_drs
1    0.953
2    0.034
0    0.007
3    0.006
Name: proportion, dtype: float64

In [7]:
patients_df['no_of_drs'].sum()

np.int64(1039)

In [8]:
import pandas as pd
import random
from datetime import datetime, timedelta


def generate_diagnostic_report():
    diagnostic_reports = []

    # Define date range
    start_date = datetime(2019, 1, 1)
    end_date = datetime(2025, 12, 31)
    days_between = (end_date - start_date).days

    for i, row in patients_df.iterrows():

        for j in range(row['no_of_drs']):

            patient_id = row['patient_id']

            random_days = random.randint(0, days_between)
            created_date = (start_date + timedelta(days=random_days)).date()

            diagnostic_reports.append({
                "report_id": f"{patient_id}-{j:02d}",
                "patient_id": patient_id,
                "created_date": created_date,
            })

    df = pd.DataFrame(diagnostic_reports)
    return df

diagnostic_reports_df = generate_diagnostic_report()
diagnostic_reports_df

,report_id,patient_id,created_date
0,0-00,0,2019-07-27
1,1-00,1,2025-04-15
2,1-01,1,2019-07-02
3,2-00,2,2021-05-13
4,3-00,3,2020-05-11
...,...,...,...
1034,996-00,996,2024-09-11
1035,997-00,997,2025-05-08
1036,997-01,997,2020-09-06
1037,998-00,998,2022-05-11


In [9]:
import os
from pathlib import Path


save_dir = Path(os.environ['HIE_DASHBOARD_BASE'], 'data', 'v_1.0')
save_dir.mkdir(parents=True, exist_ok=True)

diagnostic_reports_df.to_csv(Path(save_dir, 'diagnostic_reports.csv'), index=False)

## Blood Culture Observation

In [11]:
import os
from pathlib import Path

import pandas as pd


save_dir = Path(os.environ['HIE_DASHBOARD_BASE'], 'data', 'v_1.0')
patients_file = Path(save_dir, 'diagnostic_reports.csv')

diagnostic_reports_df = pd.read_csv(patients_file)
diagnostic_reports_df

,report_id,patient_id,created_date
0,0-00,0,2019-07-27
1,1-00,1,2025-04-15
2,1-01,1,2019-07-02
3,2-00,2,2021-05-13
4,3-00,3,2020-05-11
...,...,...,...
1034,996-00,996,2024-09-11
1035,997-00,997,2025-05-08
1036,997-01,997,2020-09-06
1037,998-00,998,2022-05-11


In [12]:
import pandas as pd
import random
from datetime import datetime, timedelta


def generate_blood_culture_observation():
    blood_culture_observations = []

    for i, row in diagnostic_reports_df.iterrows():

        parent_id = row['report_id']

        result = random.choices(['Positive', 'Negative'], weights=[40, 60])[0]

        blood_culture_observations.append({
            "report_id": f"{parent_id}-00",
            "parent_id": parent_id,
            "result": result,
        })

    df = pd.DataFrame(blood_culture_observations)
    return df

blood_culture_observations_df = generate_blood_culture_observation()
blood_culture_observations_df

,report_id,parent_id,result
0,0-00-00,0-00,Positive
1,1-00-00,1-00,Negative
2,1-01-00,1-01,Negative
3,2-00-00,2-00,Negative
4,3-00-00,3-00,Negative
...,...,...,...
1034,996-00-00,996-00,Negative
1035,997-00-00,997-00,Negative
1036,997-01-00,997-01,Negative
1037,998-00-00,998-00,Negative


In [17]:
blood_culture_observations_df.result.value_counts()

result
Negative    621
Positive    418
Name: count, dtype: int64

In [14]:
import os
from pathlib import Path


save_dir = Path(os.environ['HIE_DASHBOARD_BASE'], 'data', 'v_1.0')
save_dir.mkdir(parents=True, exist_ok=True)

blood_culture_observations_df.to_csv(Path(save_dir, 'blood_culture_observations.csv'), index=False)

## Susceptibility Observation

In [15]:
import os
from pathlib import Path

import pandas as pd


save_dir = Path(os.environ['HIE_DASHBOARD_BASE'], 'data', 'v_1.0')
patients_file = Path(save_dir, 'blood_culture_observations.csv')

blood_culture_observations_df = pd.read_csv(patients_file)
blood_culture_observations_df

,report_id,parent_id,result
0,0-00-00,0-00,Positive
1,1-00-00,1-00,Negative
2,1-01-00,1-01,Negative
3,2-00-00,2-00,Negative
4,3-00-00,3-00,Negative
...,...,...,...
1034,996-00-00,996-00,Negative
1035,997-00-00,997-00,Negative
1036,997-01-00,997-01,Negative
1037,998-00-00,998-00,Negative


In [19]:
import pandas as pd
import random
from datetime import datetime, timedelta


def generate_susceptibility_observation():
    blood_culture_observations = []

    for i, row in blood_culture_observations_df[blood_culture_observations_df.result == 'Positive'].iterrows():

        antibiotics = ['Amoxicillin', 'Piperacillin', 'Ceftriaxone', 'Ceftazidime', 'Cefepime', 'Ertapenem', 'Imipenem', 'Meropenem', 'Ciprofloxacin', 'Levofloxacin', 'Gentamicin', 'Tobramycin', 'Amikacin']

        for antibiotic in antibiotics:

            parent_id = row['report_id']

            result = random.choices(['Susceptible', 'Intermediate', 'Resistant'], weights=[40, 5, 55])[0]

            blood_culture_observations.append({
                "report_id": f"{parent_id}-00",
                "parent_id": parent_id,
                "antibiotic": antibiotic,
                "result": result,
            })

    df = pd.DataFrame(blood_culture_observations)
    return df

susceptibility_observation_df = generate_susceptibility_observation()
susceptibility_observation_df

,report_id,parent_id,antibiotic,result
0,0-00-00-00,0-00-00,Amoxicillin,Susceptible
1,0-00-00-00,0-00-00,Piperacillin,Resistant
2,0-00-00-00,0-00-00,Ceftriaxone,Resistant
3,0-00-00-00,0-00-00,Ceftazidime,Resistant
4,0-00-00-00,0-00-00,Cefepime,Resistant
...,...,...,...,...
5429,999-00-00-00,999-00-00,Ciprofloxacin,Resistant
5430,999-00-00-00,999-00-00,Levofloxacin,Resistant
5431,999-00-00-00,999-00-00,Gentamicin,Susceptible
5432,999-00-00-00,999-00-00,Tobramycin,Resistant


In [20]:
import os
from pathlib import Path


save_dir = Path(os.environ['HIE_DASHBOARD_BASE'], 'data', 'v_1.0')
save_dir.mkdir(parents=True, exist_ok=True)

susceptibility_observation_df.to_csv(Path(save_dir, 'susceptibility_observations.csv'), index=False)

## Molecular Sequence